In [3]:
import pandas as pd

url = "https://drive.google.com/uc?id=1u2DE9aWZrLfdl3ZpHtiLa4RPO8RmgnHY"
df = pd.read_csv(url)
df.head(10)

,Transaction_ID,Customer_ID,Customer_Name,City,State,Country,Age,Gender,Order_Date,Product_Name,Quantity,Unit_Price,Total_Amount,Product_Category,Product_Brand
0,8691788.0,37249.0,Michelle Harrington,Dortmund,Berlin,Germany,21.0,Male,18-09-2023,Cycling shorts,3.0,108.028758,324.086273,Clothing,Nike
1,2174773.0,69749.0,Kelsey Hill,Nottingham,England,UK,19.0,Female,31-12-2023,Lenovo Tab,2.0,403.353912,806.707825,Electronics,Samsung
2,6679610.0,30192.0,Scott Jensen,Geelong,New South Wales,Australia,48.0,Male,26-04-2023,Sports equipment,3.0,354.477580,1063.432739,Books,Penguin Books
3,7232460.0,62101.0,Joseph Miller,Edmonton,Ontario,Canada,56.0,Male,05-08-2023,Utility knife,7.0,352.407715,2466.854004,Home Decor,Home Depot
4,4983775.0,27901.0,Debra Coleman,Bristol,England,UK,22.0,Male,01-10-2024,Chocolate cookies,2.0,124.276527,248.553055,Grocery,Nestle
5,6095326.0,41289.0,Ryan Johnson,Brisbane,New South Wales,Australia,58.0,Female,21-09-2023,Lenovo Tab,4.0,296.291809,1185.167236,Electronics,Apple
6,5434096.0,97285.0,Erin Lewis,Kitchener,Ontario,Canada,29.0,Female,26-06-2023,QLED TV,2.0,315.057648,630.115295,Electronics,Samsung
7,2344675.0,26603.0,Angela Fields,Munich,Berlin,Germany,29.0,Male,24-03-2023,Dress shirt,1.0,46.588070,46.588070,Clothing,Zara
8,4155845.0,80175.0,Diane Clark,Wollongong,New South Wales,Australia,46.0,Male,01-06-2024,Dark chocolate,8.0,328.839294,2630.714355,Grocery,Nestle
9,4926148.0,31878.0,Lori Bell,Cologne,Berlin,Germany,25.0,Male,10-04-2023,Candles,10.0,397.611230,3976.112305,Home Decor,Home Depot


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 302010 entries, 0 to 302009
Data columns (total 15 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Transaction_ID    301677 non-null  float64
 1   Customer_ID       301702 non-null  float64
 2   Customer_Name     301628 non-null  str    
 3   City              301762 non-null  str    
 4   State             301729 non-null  str    
 5   Country           301739 non-null  str    
 6   Age               301837 non-null  float64
 7   Gender            301693 non-null  str    
 8   Order_Date        301651 non-null  str    
 9   Product_Name      302010 non-null  str    
 10  Quantity          301649 non-null  float64
 11  Unit_Price        301299 non-null  float64
 12  Total_Amount      301660 non-null  float64
 13  Product_Category  301401 non-null  str    
 14  Product_Brand     301412 non-null  str    
dtypes: float64(6), str(9)
memory usage: 34.6 MB


In [8]:
# df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Order_Date'] = pd.to_datetime(df['Order_Date'], dayfirst=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 302010 entries, 0 to 302009
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Transaction_ID    301677 non-null  float64       
 1   Customer_ID       301702 non-null  float64       
 2   Customer_Name     301628 non-null  str           
 3   City              301762 non-null  str           
 4   State             301729 non-null  str           
 5   Country           301739 non-null  str           
 6   Age               301837 non-null  float64       
 7   Gender            301693 non-null  str           
 8   Order_Date        301651 non-null  datetime64[us]
 9   Product_Name      302010 non-null  str           
 10  Quantity          301649 non-null  float64       
 11  Unit_Price        301299 non-null  float64       
 12  Total_Amount      301660 non-null  float64       
 13  Product_Category  301401 non-null  str           
 14  Product_Brand  

In [11]:
import datetime as dt

snapshot_date = df['Date'].max() + dt.timedelta(days=1)
snapshot_date

Timestamp('2024-12-03 00:00:00')

In [12]:
rfm = df.groupby('Customer_ID').agg({
    'Order_Date': lambda x: (snapshot_date - x.max()).days,   # Recency
    'Customer_ID': 'count',                            # Frequency
    'Total_Amount': 'sum'                              # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

,Recency,Frequency,Monetary
Customer_ID,,,
10000.0,380.0,4,5007.566467
10001.0,382.0,5,8136.462738
10002.0,372.0,5,4104.014053
10003.0,505.0,2,2340.496399
10004.0,308.0,2,2356.516663


In [14]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

rfm.head()

,Recency,Frequency,Monetary,R_score,F_score,M_score
Customer_ID,,,,,,
10000.0,380.0,4,5007.566467,2,3,3
10001.0,382.0,5,8136.462738,2,4,5
10002.0,372.0,5,4104.014053,3,4,3
10003.0,505.0,2,2340.496399,1,1,2
10004.0,308.0,2,2356.516663,4,1,2


In [16]:
rfm[['R_score','F_score','M_score']].isnull().sum()

R_score    0
F_score    0
M_score    0
dtype: int64

In [15]:
rfm = rfm.dropna(subset=['R_score','F_score','M_score'])

In [17]:
rfm['RFM_score'] = (
    rfm['R_score'].astype(int).astype(str) +
    rfm['F_score'].astype(int).astype(str) +
    rfm['M_score'].astype(int).astype(str)
)
rfm.head()

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_score
Customer_ID,,,,,,,
10000.0,380.0,4,5007.566467,2,3,3,233
10001.0,382.0,5,8136.462738,2,4,5,245
10002.0,372.0,5,4104.014053,3,4,3,343
10003.0,505.0,2,2340.496399,1,1,2,112
10004.0,308.0,2,2356.516663,4,1,2,412


In [21]:
def segment_customer(row):

    # Champions
    if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
        return 'Champions'

    # Loyal Customers
    elif row['F_score'] >= 4 and row['R_score'] >= 3:
        return 'Loyal Customers'

    # Can't Lose Them
    elif row['R_score'] <= 2 and row['F_score'] >= 4:
        return "Can't Lose Them"

    # At Risk
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return 'At Risk'

    # Hibernating
    elif row['R_score'] <= 2 and row['F_score'] <= 2:
        return 'Hibernating'

    # Lost
    elif row['R_score'] == 1 and row['F_score'] == 1:
        return 'Lost'

    # Potential Loyalist
    elif row['R_score'] >= 4 and row['F_score'] >= 2:
        return 'Potential Loyalist'

    # Recent Users
    elif row['R_score'] == 5 and row['F_score'] == 1:
        return 'Recent Users'

    # Promising
    elif row['R_score'] == 4 and row['F_score'] == 1:
        return 'Promising'

    # Needs Attention
    elif row['R_score'] == 3:
        return 'Needs Attention'

    # About To Sleep
    elif row['R_score'] == 2:
        return 'About To Sleep'

    # Price Sensitive (low monetary)
    elif row['M_score'] <= 2:
        return 'Price Sensitive'

    else:
        return 'Others'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm.head()

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_score,Segment
Customer_ID,,,,,,,,
10000.0,380.0,4,5007.566467,2,3,3,233,At Risk
10001.0,382.0,5,8136.462738,2,4,5,245,Can't Lose Them
10002.0,372.0,5,4104.014053,3,4,3,343,Loyal Customers
10003.0,505.0,2,2340.496399,1,1,2,112,Hibernating
10004.0,308.0,2,2356.516663,4,1,2,412,Promising


In [26]:
rfm.to_excel("Rfm_output.xlsx",engine="xlsxwriter", index=False)

In [25]:
rfm.shape

(86753, 8)